# YOLOv8 Training and Inference with optimization

This notebook demonstrates how to train a modified YOLOv8 model on a custom dataset, perform inference, and optimize the model using TensorRT for faster inference. The following steps will be covered:
1. Mount Google Drive
2. Set up the YOLOv8 environment
3. Train YOLOv8 models (original and modified)
4. Perform inference with trained models
5. Optimize the YOLOv8 model using TensorRT
6. Perform inference with TensorRT-optimized model

## 1. Mount Google Drive
First, we mount Google Drive to access datasets and save results.

In [ ]:
from google.colab import drive
drive.mount('/content/gdrive')

## 2. Set Up YOLOv8 Environment
We need to create a directory for the YOLOv8 project and install the required libraries.

In [ ]:
import os

# Static variables
ROOT_DIR = '/content/gdrive/MyDrive'
YOLOV8_DIR = 'yolov8-architecture'
YOLOV8_CONFIG_URL = 'https://raw.githubusercontent.com/ultralytics/ultralytics/main/ultralytics/cfg/models/v8/yolov8.yaml'
DATA_FILE_PATH = os.path.join(ROOT_DIR, 'ship.yaml')
TEST_IMAGE_PATH = os.path.join(ROOT_DIR, 'data_example/images/test/Gao_ship_hh_02016082544030206.jpg')

# Change to the desired working directory
os.chdir(ROOT_DIR)

# Create YOLOv8 root folder
if not os.path.exists(YOLOV8_DIR):
    os.makedirs(YOLOV8_DIR)

# Navigate to the YOLOv8 root folder
os.chdir(YOLOV8_DIR)

# Install YOLOv8
!pip install ultralytics

import ultralytics
ultralytics.checks()

## 3. Prepare YOLOv8 Configuration
Download the YOLOv8 architecture file and create configuration files for different models.

In [ ]:
import shutil

# Static variables for configuration files
SOURCE_FILE = 'yolov8.yaml'
DESTINATION_FILE_1 = 'yolov8s.yaml'
DESTINATION_FILE_2 = 'yolov8s-small.yaml'

# Download the YOLOv8 Architecture File
!wget {YOLOV8_CONFIG_URL} -O {SOURCE_FILE}

# Copy the configuration files
if not os.path.exists(DESTINATION_FILE_1):
    shutil.copyfile(SOURCE_FILE, DESTINATION_FILE_1)

if not os.path.exists(DESTINATION_FILE_2):
    shutil.copyfile(SOURCE_FILE, DESTINATION_FILE_2)

## 4. Train YOLOv8 Models
Train the original and modified YOLOv8 models using the custom dataset.

In [ ]:
# Static variables for training configurations
WORKERS = 8
BATCH_SIZE = 64
DEVICE = 0
EPOCHS = 200
PATIENCE = 50
TRAIN_NAME_1 = 'yolov8_ships'
TRAIN_NAME_2 = 'yolov8_ships_small'
OPTIMIZER = 'SGD'
LEARNING_RATE = 0.01
WEIGHT_DECAY = 0.001
WARMUP = 1
MOMENTUM = 0.95

# Training Original Model
!yolo detect train model={DESTINATION_FILE_1} data={DATA_FILE_PATH} workers={WORKERS} batch={BATCH_SIZE} device={DEVICE} epochs={EPOCHS} patience={PATIENCE} name={TRAIN_NAME_1} optimizer={OPTIMIZER} lr0={LEARNING_RATE} weight_decay={WEIGHT_DECAY} warmup_epochs={WARMUP} momentum={MOMENTUM}

# Training Modified Model (Small Objects)
!yolo detect train model={DESTINATION_FILE_2} data={DATA_FILE_PATH} workers={WORKERS} batch={BATCH_SIZE} device={DEVICE} epochs={EPOCHS} patience={PATIENCE} name={TRAIN_NAME_2} optimizer={OPTIMIZER} lr0={LEARNING_RATE} weight_decay={WEIGHT_DECAY} warmup_epochs={WARMUP} momentum={MOMENTUM}

## 5. Perform Inference with Trained Models
Use the trained models to perform inference on a test image.

In [ ]:
# Static variables for inference
PREDICT_NAME_1 = 'runs/detect/yolov8_ships/weights/best.pt'
PREDICT_NAME_2 = 'runs/detect/yolov8_ships_small/weights/best.pt'

# Original YOLOv8 model
!yolo detect predict model={PREDICT_NAME_1} source={TEST_IMAGE_PATH} save=True

# Modified YOLOv8 model
!yolo detect predict model={PREDICT_NAME_2} source={TEST_IMAGE_PATH} save=True

## 6. Visualize Inference Results
Display the results of the inference.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

def show_image(image_path):
    img = mpimg.imread(image_path)
    plt.imshow(img)
    plt.axis('off')
    plt.show()

# Display results for original YOLOv8 model
show_image("runs/detect/predict/Gao_ship_hh_02016082544030206.jpg")

# Display results for modified YOLOv8 model
show_image("runs/detect/predict2/Gao_ship_hh_02016082544030206.jpg")

## 7. Optimize YOLOv8 Model Using TensorRT
Install necessary libraries and export the YOLOv8 model to TensorRT format.

In [ ]:
# Install required libraries
!pip install tensorrt tensorrt_lean tensorrt_dispatch
!pip install onnx onnxsim onnxruntime-gpu

# Static variable for TensorRT export
EXPORT_NAME = 'runs/detect/yolov8_ships_small/weights/best.pt'

# Export YOLOv8 Model to TensorRT
!yolo export model={EXPORT_NAME} format=engine half=True device=0

## 8. Evaluate TensorRT-Optimized Model
Validate the TensorRT-optimized model and perform inference.

In [ ]:
# Static variables for validation
VAL_NAME_1 = 'runs/detect/yolov8_ships/weights/best.pt'
VAL_NAME_2 = 'runs/detect/yolov8_ships_small/weights/best.engine'
IOU = 0.5
VAL_NAME_3 = 'yolov8_ships_val'
VAL_NAME_4 = 'yolov8_ships_small_tensorrt_val'

# Validate original YOLOv8 model
!yolo detect val model={VAL_NAME_1} data={DATA_FILE_PATH} iou={IOU} name={VAL_NAME_3}

# Validate TensorRT-optimized YOLOv8 model
!yolo detect val model={VAL_NAME_2} data={DATA_FILE_PATH} iou={IOU} name={VAL_NAME_4}

# Perform inference with original YOLOv8 model
!yolo detect predict model={VAL_NAME_1} source={TEST_IMAGE_PATH} save=True

# Perform inference with TensorRT-optimized YOLOv8 model
!yolo detect predict model={VAL_NAME_2} source={TEST_IMAGE_PATH} save=True